# 03 — Segmentation Study

Explore how detector segmentation (nSections) affects energy-loss
resolution, hit multiplicity, and Landau fit quality.

This notebook covers:
1. Loading all segmentation files for a beam type
2. Per-segmentation Edep distributions
3. Landau overlay with fit parameters vs nSec
4. nHits distributions and zero-hit fractions
5. Comparing Krakow vs Muon segmentation trends

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from hibeam import config
from hibeam.io import seg_loader
from hibeam.physics import fitting
from hibeam.plotting import style, segmentation_plots

cfg = config.load('../config.yaml')
style.apply(cfg)

## 1. Load segmentation data

In [ ]:
# ── Change this to switch beam type ────────────────────────────────
beam = 'krakow'   # Options: 'krakow', 'muon'
# ───────────────────────────────────────────────────────────────────

dir_key = f'{beam}_dir'
seg_dir = config.resolve_path(cfg, cfg.paths.segmentation[dir_key])

records = seg_loader.load(
    str(seg_dir),
    pattern=cfg.segmentation.pattern,
    tree_name=cfg.segmentation.tree_name,
    edep_branch=cfg.segmentation.branch,
    min_steps=cfg.segmentation.min_steps,
    edep_floor_mev=cfg.segmentation.edep_floor_mev,
)
print(f'\nLoaded {len(records)} segmentation files for {beam}')

## 2. Raw Edep distributions per segmentation

In [ ]:
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(records)))
fig, ax = plt.subplots(figsize=(12, 6))

for idx, rec in enumerate(records):
    edep = rec['edep_per_event']
    if edep.max() < 1.0:
        edep = edep * 1000  # GeV → MeV
    edep = edep[edep > 0]
    if len(edep) == 0:
        continue
    bins = np.linspace(0, np.percentile(edep, 98), 60)
    ax.hist(edep, bins=bins, histtype='step', lw=1.2,
            color=cmap[idx], label=f'nSec={rec["nsec"]}')

ax.set_xlabel(r'$\sum E_\mathrm{dep}$ [MeV]')
ax.set_ylabel('Events / bin')
ax.set_title(f'Per-event Edep — {beam}')
ax.legend(fontsize=7, ncol=3)
plt.show()

## 3. Landau overlay with fit parameters

In [ ]:
fig, fit_results = segmentation_plots.overlay_segmentations(
    records,
    truncation=cfg.fitting.truncation,
    n_bins=cfg.segmentation.fit_nbins,
    edep_floor_mev=cfg.segmentation.edep_floor_mev,
    title=f'Segmentation overlay — {beam}',
    cfg=cfg,
)
plt.show()

# Print summary table
print(f'\n{"nSec":>6}  {"MPV [MeV]":>10}  {"ξ [MeV]":>10}  {"χ²/ndf":>8}  {"p-value":>8}')
print('-' * 55)
for fr in fit_results:
    r = fr['result']
    print(f'{fr["nsec"]:>6}  {r.mpv:>10.4f}  {r.scale:>10.4f}  {r.chi2_red:>8.3f}  {r.p_value:>8.3f}')

## 4. nHits vs nSections

In [ ]:
fig = segmentation_plots.plot_nhits_vs_nsec(
    records,
    title=f'nHits vs segmentation — {beam}',
    cfg=cfg,
)
plt.show()

## 5. nHits distributions

In [ ]:
fig = segmentation_plots.plot_nhits_distribution(
    records,
    title=f'nHits per event — {beam}',
    cfg=cfg,
)
plt.show()

## 6. Fit a single segmentation with custom parameters

Pick a specific nSec and experiment with fit settings.

In [ ]:
# ── Change these ──────────────────────────────────────────────────
target_nsec = 10
custom_truncation = 0.70
custom_bins = 40
# ──────────────────────────────────────────────────────────────────

rec = next((r for r in records if r['nsec'] == target_nsec), None)
if rec is None:
    print(f'nSec={target_nsec} not found. Available: {[r["nsec"] for r in records]}')
else:
    edep = rec['edep_per_event'].copy()
    if edep.max() < 1.0:
        edep = edep * 1000
    edep = edep[edep > cfg.segmentation.edep_floor_mev]

    result = fitting.fit_landau(edep, truncation=custom_truncation, n_bins=custom_bins)

    from hibeam.plotting import histograms
    from hibeam.utils import print_fit_summary

    print_fit_summary(result.as_dict())
    histograms.plot_dedx(
        result,
        title=f'{beam} nSec={target_nsec} — trunc={custom_truncation}, bins={custom_bins}',
        cfg=cfg,
    )

## 7. Compare Krakow vs Muon trends

Load both beam types and plot MPV vs nSec side by side.

In [ ]:
both = {}
for b in ['krakow', 'muon']:
    d = config.resolve_path(cfg, cfg.paths.segmentation[f'{b}_dir'])
    if d.is_dir():
        both[b] = seg_loader.load(
            str(d), pattern=cfg.segmentation.pattern,
            min_steps=cfg.segmentation.min_steps,
            edep_floor_mev=cfg.segmentation.edep_floor_mev,
        )

if len(both) == 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = style.get_colors(cfg)

    for ax, (b, recs) in zip([ax1, ax2], both.items()):
        nsecs, mpvs = [], []
        for rec in recs:
            edep = rec['edep_per_event'].copy()
            if edep.max() < 1.0:
                edep = edep * 1000
            edep = edep[edep > cfg.segmentation.edep_floor_mev]
            if len(edep) < 50:
                continue
            try:
                r = fitting.fit_landau(edep, truncation=0.70, n_bins=40)
                nsecs.append(rec['nsec'])
                mpvs.append(r.mpv)
            except:
                pass
        ax.plot(nsecs, mpvs, 'o-', color=colors[0])
        ax.set_xlabel('nSections')
        ax.set_ylabel('MPV [MeV]')
        ax.set_title(f'{b.capitalize()}')

    fig.suptitle('MPV vs segmentation — both beam types', fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print('Need both krakow and muon segmentation directories')